#Reglas generales

**Regla 1 — Una celda, una responsabilidad.** Cada celda hace una sola cosa. Si necesitas un comentario largo para explicar qué hace, divídela en dos.

**Regla 2 — Comentario de autor y fecha en cada celda nueva.**
`# [Nombre] [DD/MM] — descripción breve de qué hace esta celda`

**Regla 3 — Avisar antes de editar o ejecutar.** Si alguien edita mientras otro ejecuta, el notebook se puede corromper. Avisar en el canal antes de entrar.

**Regla 4 — Nunca borrar código que funciona.** Comentar el bloque anterior y escribir el nuevo debajo. Solo el líder de desarrollo autoriza borrado definitivo.

**Regla 5 — El notebook debe correr limpio de arriba a abajo.** Antes de marcar algo como listo: Runtime → Restart and run all. Si falla, no está listo.

**Regla 6 — No mezclar entrenamiento con inferencia.** Este notebook solo entrena y guarda. Nada de inferencia acá.

**Regla 7 — Todas las constantes van arriba.** Rutas, nombres de archivos, hiperparámetros. Todo en la sección de configuración al inicio. Nada hardcodeado en medio del código.

**Regla 8 — Las celdas de prueba van al final.** Marcadas con `# [PRUEBA]`. No se mezclan con el pipeline principal.

**Regla 9 — Los modelos guardados se versionan.** Nunca sobreescribir un modelo sin respaldo.

**Regla 10 — Un responsable principal por notebook.** El dueño del notebook aprueba cambios al pipeline principal.

In [ ]:
#RUNEAR ESTO PARA ACCESO A DRIVE

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Validación y limpieza del dataset

Estas celdas recorren las carpetas `Train` y `Test`, detectan imágenes corruptas/truncadas o con dimensiones distintas a `96x96`, calculan hash SHA256 para detectar duplicados y generan reportes CSV.

En esta versión, `EJECUTAR_LIMPIEZA = True` y `MOVER_A_CUARENTENA = False`, por lo que las imágenes inválidas y los duplicados detectados se eliminan definitivamente del dataset. Si no hay imágenes inválidas o duplicadas, el notebook lo informa y no elimina nada.



In [ ]:
# [Benjamin Marti] [18/06/2026] — Configuración para validar imágenes y eliminar duplicados/descartadas del dataset

import os

BASE_PATH = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO"

# El notebook actual trabaja con estas carpetas originales.
SPLITS_A_REVISAR = ["Train", "Test"]

# Formatos de imagen aceptados.
IMG_EXTS = (".jpg", ".jpeg", ".png")

# Tamaño requerido por el proyecto.
TAMANO_ESPERADO = (96, 96)

# Carpeta donde se guardarán los CSV de control.
REPORTES_DIR = os.path.join(BASE_PATH, "reportes_limpieza")
os.makedirs(REPORTES_DIR, exist_ok=True)

# IMPORTANTE:
# True = aplica la limpieza: elimina imágenes inválidas y duplicados detectados.
# False = solo genera reportes, no modifica archivos.
EJECUTAR_LIMPIEZA = True

# False = elimina definitivamente los archivos del dataset.
# True  = los mueve a cuarentena en vez de borrarlos.
# Como necesitas que se eliminen, queda en False.
MOVER_A_CUARENTENA = False

CUARENTENA_DIR = os.path.join(REPORTES_DIR, "cuarentena")
if MOVER_A_CUARENTENA:
    os.makedirs(CUARENTENA_DIR, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("Existe BASE_PATH:", os.path.exists(BASE_PATH))
print("Reportes en:", REPORTES_DIR)
print("EJECUTAR_LIMPIEZA:", EJECUTAR_LIMPIEZA)
print("MOVER_A_CUARENTENA:", MOVER_A_CUARENTENA)

if EJECUTAR_LIMPIEZA and not MOVER_A_CUARENTENA:
    print("ATENCIÓN: la limpieza eliminará definitivamente las imágenes inválidas y duplicadas detectadas.")
elif EJECUTAR_LIMPIEZA and MOVER_A_CUARENTENA:
    print("La limpieza moverá las imágenes inválidas y duplicadas a cuarentena.")
else:
    print("Modo reporte: no se eliminará ni moverá ningún archivo.")



In [ ]:
# [Benjamin Marti] [14/06/2026] — Recorre dataset, detecta corruptas/tamaño incorrecto y calcula hash SHA256

import os
import shutil
import hashlib
import pandas as pd
from PIL import Image, ImageFile, UnidentifiedImageError

# Importante: False hace que PIL lance error si la imagen está truncada.
ImageFile.LOAD_TRUNCATED_IMAGES = False

def obtener_split_y_label(ruta_archivo):
    """
    Extrae split y label desde una ruta tipo:
    BASE_PATH/Train/happy/archivo.jpg
    BASE_PATH/Test/sad/archivo.jpg
    """
    rel = os.path.relpath(ruta_archivo, BASE_PATH).replace("\\", "/")
    partes = rel.split("/")
    split = partes[0] if len(partes) > 0 else ""
    label = partes[1] if len(partes) > 1 else ""
    return split, label, rel

def calcular_sha256(ruta_archivo, tamano_bloque=1024 * 1024):
    sha256 = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        while True:
            bloque = f.read(tamano_bloque)
            if not bloque:
                break
            sha256.update(bloque)
    return sha256.hexdigest()

def revisar_imagen(ruta_imagen):
    """
    Devuelve:
    - es_valida: True/False
    - motivo: texto explicando el resultado
    - dimensiones: tupla o None
    """
    try:
        # Primer intento: verifica estructura interna del archivo.
        with Image.open(ruta_imagen) as img:
            img.verify()

        # Segundo intento: reabre y carga realmente los píxeles.
        with Image.open(ruta_imagen) as img:
            img.load()
            dimensiones = img.size

        if dimensiones != TAMANO_ESPERADO:
            return False, f"dimensiones_distintas_a_96x96 ({dimensiones[0]}x{dimensiones[1]})", dimensiones

        return True, "ok", dimensiones

    except UnidentifiedImageError:
        return False, "archivo_no_reconocido_como_imagen", None
    except OSError as e:
        return False, f"imagen_no_se_puede_abrir_o_truncada ({str(e)})", None
    except Exception as e:
        return False, f"error_lectura ({type(e).__name__}: {str(e)})", None

def listar_imagenes_dataset():
    rutas = []

    for split in SPLITS_A_REVISAR:
        split_path = os.path.join(BASE_PATH, split)

        if not os.path.isdir(split_path):
            print(f"Advertencia: no existe la carpeta {split_path}")
            continue

        for raiz, _, archivos in os.walk(split_path):
            for archivo in archivos:
                if archivo.lower().endswith(IMG_EXTS):
                    rutas.append(os.path.join(raiz, archivo))

    return sorted(rutas)

rutas_imagenes = listar_imagenes_dataset()

print("Total de imágenes encontradas:", len(rutas_imagenes))

imagenes_validas = []
imagenes_descartadas = []

for i, ruta in enumerate(rutas_imagenes, start=1):
    split, label, ruta_relativa = obtener_split_y_label(ruta)

    es_valida, motivo, dimensiones = revisar_imagen(ruta)

    if not es_valida:
        imagenes_descartadas.append({
            "ruta": ruta,
            "ruta_relativa": ruta_relativa,
            "split": split,
            "label": label,
            "motivo": motivo,
            "dimensiones": "" if dimensiones is None else f"{dimensiones[0]}x{dimensiones[1]}"
        })
        continue

    try:
        hash_sha256 = calcular_sha256(ruta)

        imagenes_validas.append({
            "ruta": ruta,
            "ruta_relativa": ruta_relativa,
            "split": split,
            "label": label,
            "dimensiones": f"{dimensiones[0]}x{dimensiones[1]}",
            "sha256": hash_sha256
        })

    except Exception as e:
        imagenes_descartadas.append({
            "ruta": ruta,
            "ruta_relativa": ruta_relativa,
            "split": split,
            "label": label,
            "motivo": f"error_calculando_hash ({type(e).__name__}: {str(e)})",
            "dimensiones": f"{dimensiones[0]}x{dimensiones[1]}"
        })

    if i % 5000 == 0:
        print(f"Procesadas {i}/{len(rutas_imagenes)} imágenes...")

df_validas = pd.DataFrame(imagenes_validas)
df_descartadas = pd.DataFrame(imagenes_descartadas)

ruta_csv_validas = os.path.join(REPORTES_DIR, "imagenes_validas_con_hash.csv")
ruta_csv_descartadas = os.path.join(REPORTES_DIR, "imagenes_descartadas.csv")

df_validas.to_csv(ruta_csv_validas, index=False, encoding="utf-8")
df_descartadas.to_csv(ruta_csv_descartadas, index=False, encoding="utf-8")

print("\n=== Resultado validación ===")
print("Imágenes válidas:", len(df_validas))
print("Imágenes descartadas:", len(df_descartadas))
print("\nCSV válidas:", ruta_csv_validas)
print("CSV descartadas:", ruta_csv_descartadas)

if len(df_descartadas) > 0:
    print("\nMotivos de descarte:")
    print(df_descartadas["motivo"].value_counts().to_string())
    display(df_descartadas.head(10))
else:
    print("\nNo se detectaron imágenes corruptas, truncadas o con dimensiones incorrectas.")


In [ ]:
# [Benjamin Marti] [14/06/2026] — Detecta duplicados por SHA256 y conserva una copia por grupo

import os
import shutil
import pandas as pd

def ruta_cuarentena_desde_relativa(ruta_relativa, prefijo):
    """
    Crea una ruta de respaldo dentro de reportes_limpieza/cuarentena
    manteniendo la estructura relativa original.
    """
    destino = os.path.join(CUARENTENA_DIR, prefijo, ruta_relativa)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    return destino

duplicados_detectados = []

if len(df_validas) == 0:
    print("No hay imágenes válidas para revisar duplicados.")
else:
    grupos_hash = df_validas.groupby("sha256", sort=False)

    grupo_id = 0

    for hash_img, grupo in grupos_hash:
        if len(grupo) <= 1:
            continue

        grupo_id += 1

        # Orden determinístico: se conserva la primera ruta ordenada.
        grupo_ordenado = grupo.sort_values("ruta").reset_index(drop=True)
        conservada = grupo_ordenado.iloc[0]

        for _, duplicada in grupo_ordenado.iloc[1:].iterrows():
            duplicados_detectados.append({
                "grupo_duplicado": grupo_id,
                "sha256": hash_img,
                "ruta_conservada": conservada["ruta"],
                "ruta_conservada_relativa": conservada["ruta_relativa"],
                "split_conservado": conservada["split"],
                "label_conservado": conservada["label"],
                "ruta_duplicada": duplicada["ruta"],
                "ruta_duplicada_relativa": duplicada["ruta_relativa"],
                "split_duplicado": duplicada["split"],
                "label_duplicado": duplicada["label"],
                "accion": "pendiente"
            })

df_duplicados = pd.DataFrame(duplicados_detectados)

ruta_csv_duplicados = os.path.join(REPORTES_DIR, "duplicados_detectados.csv")
df_duplicados.to_csv(ruta_csv_duplicados, index=False, encoding="utf-8")

print("Grupos de duplicados detectados:", df_duplicados["grupo_duplicado"].nunique() if len(df_duplicados) > 0 else 0)
print("Archivos duplicados a retirar:", len(df_duplicados))
print("CSV duplicados:", ruta_csv_duplicados)

if len(df_duplicados) > 0:
    print("\nPrimeros duplicados detectados:")
    display(df_duplicados.head(10))
else:
    print("\nNo se detectaron duplicados idénticos por SHA256.")


In [ ]:
# [Benjamin Marti] [18/06/2026] — Elimina imágenes inválidas y duplicadas, dejando registro CSV y conteo final

import os
import shutil
import pandas as pd

acciones_realizadas = []

def ruta_cuarentena_desde_relativa(ruta_relativa, prefijo):
    """
    Crea una ruta de respaldo dentro de reportes_limpieza/cuarentena
    manteniendo la estructura relativa original.
    """
    destino = os.path.join(CUARENTENA_DIR, prefijo, ruta_relativa)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    return destino

def mover_o_eliminar_archivo(ruta_origen, ruta_relativa, categoria):
    """
    Si MOVER_A_CUARENTENA=True, mueve el archivo a reportes_limpieza/cuarentena.
    Si MOVER_A_CUARENTENA=False, lo elimina definitivamente.
    """
    if not os.path.exists(ruta_origen):
        return "no_existe", ""

    if MOVER_A_CUARENTENA:
        ruta_destino = ruta_cuarentena_desde_relativa(ruta_relativa, categoria)

        # Evita sobrescribir si ya existe un archivo con el mismo nombre en cuarentena.
        if os.path.exists(ruta_destino):
            base, ext = os.path.splitext(ruta_destino)
            contador = 1
            while os.path.exists(f"{base}_{contador}{ext}"):
                contador += 1
            ruta_destino = f"{base}_{contador}{ext}"

        shutil.move(ruta_origen, ruta_destino)
        return "movida_a_cuarentena", ruta_destino

    os.remove(ruta_origen)
    return "eliminada_definitivamente", ""

# Valores base para que el resumen funcione incluso si no se aplica limpieza.
imagenes_invalidas_detectadas = len(df_descartadas)
duplicados_detectados_cantidad = len(df_duplicados)
imagenes_invalidas_eliminadas = 0
duplicados_eliminados = 0
imagenes_invalidas_movidas = 0
duplicados_movidos = 0
total_eliminadas = 0
total_movidas = 0
total_no_existian = 0

if not EJECUTAR_LIMPIEZA:
    print("Modo reporte: no se movió ni eliminó ningún archivo.")
    print("Para aplicar la limpieza, cambia EJECUTAR_LIMPIEZA = True en la celda de configuración.")
else:
    print("Aplicando limpieza del dataset...")

    # 1. Eliminar o mover imágenes inválidas/corruptas/truncadas/con dimensiones incorrectas.
    if len(df_descartadas) == 0:
        print("No se detectaron imágenes inválidas, corruptas, truncadas o con dimensiones distintas a 96x96.")
    else:
        for _, fila in df_descartadas.iterrows():
            estado, destino = mover_o_eliminar_archivo(
                ruta_origen=fila["ruta"],
                ruta_relativa=fila["ruta_relativa"],
                categoria="imagenes_descartadas"
            )

            acciones_realizadas.append({
                "tipo": "imagen_descartada",
                "ruta_origen": fila["ruta"],
                "ruta_relativa": fila["ruta_relativa"],
                "motivo": fila["motivo"],
                "estado": estado,
                "ruta_destino_cuarentena": destino
            })

    # 2. Eliminar o mover duplicados conservando una sola copia por grupo.
    if len(df_duplicados) == 0:
        print("No se detectaron duplicados idénticos por SHA256. No se elimina ningún duplicado.")
    else:
        for _, fila in df_duplicados.iterrows():
            estado, destino = mover_o_eliminar_archivo(
                ruta_origen=fila["ruta_duplicada"],
                ruta_relativa=fila["ruta_duplicada_relativa"],
                categoria="duplicados"
            )

            acciones_realizadas.append({
                "tipo": "duplicado",
                "ruta_origen": fila["ruta_duplicada"],
                "ruta_relativa": fila["ruta_duplicada_relativa"],
                "motivo": f"duplicado_sha256_conservada={fila['ruta_conservada_relativa']}",
                "estado": estado,
                "ruta_destino_cuarentena": destino
            })

df_acciones = pd.DataFrame(
    acciones_realizadas,
    columns=[
        "tipo",
        "ruta_origen",
        "ruta_relativa",
        "motivo",
        "estado",
        "ruta_destino_cuarentena"
    ]
)

ruta_csv_acciones = os.path.join(REPORTES_DIR, "acciones_limpieza_realizadas.csv")
df_acciones.to_csv(ruta_csv_acciones, index=False, encoding="utf-8")

if len(df_acciones) > 0:
    imagenes_invalidas_eliminadas = len(df_acciones[
        (df_acciones["tipo"] == "imagen_descartada") &
        (df_acciones["estado"] == "eliminada_definitivamente")
    ])

    duplicados_eliminados = len(df_acciones[
        (df_acciones["tipo"] == "duplicado") &
        (df_acciones["estado"] == "eliminada_definitivamente")
    ])

    imagenes_invalidas_movidas = len(df_acciones[
        (df_acciones["tipo"] == "imagen_descartada") &
        (df_acciones["estado"] == "movida_a_cuarentena")
    ])

    duplicados_movidos = len(df_acciones[
        (df_acciones["tipo"] == "duplicado") &
        (df_acciones["estado"] == "movida_a_cuarentena")
    ])

    total_no_existian = len(df_acciones[df_acciones["estado"] == "no_existe"])

total_eliminadas = imagenes_invalidas_eliminadas + duplicados_eliminados
total_movidas = imagenes_invalidas_movidas + duplicados_movidos

print("\n=== Resultado de limpieza ===")
print("Imágenes inválidas detectadas:", imagenes_invalidas_detectadas)
print("Duplicados detectados a retirar:", duplicados_detectados_cantidad)

if MOVER_A_CUARENTENA:
    print("Imágenes inválidas movidas a cuarentena:", imagenes_invalidas_movidas)
    print("Duplicados movidos a cuarentena:", duplicados_movidos)
    print("Total imágenes retiradas del dataset:", total_movidas)
else:
    print("Imágenes inválidas eliminadas:", imagenes_invalidas_eliminadas)
    print("Duplicados eliminados:", duplicados_eliminados)
    print("Total imágenes eliminadas:", total_eliminadas)

print("Archivos que ya no existían al intentar limpiar:", total_no_existian)
print("CSV de acciones:", ruta_csv_acciones)

if len(df_acciones) > 0:
    print("\nResumen por estado:")
    print(df_acciones["estado"].value_counts().to_string())
    display(df_acciones.head(10))
else:
    print("No había imágenes inválidas ni duplicados para eliminar.")



In [ ]:
# [Benjamin Marti] [18/06/2026] — Verificación posterior y resumen final para informe

import os
import pandas as pd

# Verifica que las rutas que debían retirarse ya no estén en Train/Test.
invalidas_aun_presentes = 0
duplicados_aun_presentes = 0
copias_conservadas_presentes = 0

if len(df_descartadas) > 0 and "ruta" in df_descartadas.columns:
    invalidas_aun_presentes = sum(os.path.exists(ruta) for ruta in df_descartadas["ruta"])

if len(df_duplicados) > 0:
    duplicados_aun_presentes = sum(os.path.exists(ruta) for ruta in df_duplicados["ruta_duplicada"])
    copias_conservadas_presentes = sum(os.path.exists(ruta) for ruta in df_duplicados["ruta_conservada"].drop_duplicates())

resumen = {
    "total_imagenes_revisadas": [len(rutas_imagenes)],
    "imagenes_validas_96x96": [len(df_validas)],
    "imagenes_descartadas_detectadas": [len(df_descartadas)],
    "grupos_duplicados_detectados": [df_duplicados["grupo_duplicado"].nunique() if len(df_duplicados) > 0 else 0],
    "archivos_duplicados_detectados_a_retirar": [len(df_duplicados)],
    "limpieza_aplicada": [EJECUTAR_LIMPIEZA],
    "modo_cuarentena": [MOVER_A_CUARENTENA],
    "imagenes_invalidas_eliminadas": [imagenes_invalidas_eliminadas],
    "duplicados_eliminados": [duplicados_eliminados],
    "total_imagenes_eliminadas": [total_eliminadas],
    "imagenes_invalidas_movidas_a_cuarentena": [imagenes_invalidas_movidas],
    "duplicados_movidos_a_cuarentena": [duplicados_movidos],
    "total_imagenes_retiradas_del_dataset": [total_eliminadas if not MOVER_A_CUARENTENA else total_movidas],
    "invalidas_aun_presentes_en_dataset": [invalidas_aun_presentes],
    "duplicados_aun_presentes_en_dataset": [duplicados_aun_presentes],
    "copias_conservadas_presentes": [copias_conservadas_presentes],
    "csv_descartadas": [os.path.join(REPORTES_DIR, "imagenes_descartadas.csv")],
    "csv_duplicados": [os.path.join(REPORTES_DIR, "duplicados_detectados.csv")],
    "csv_acciones": [os.path.join(REPORTES_DIR, "acciones_limpieza_realizadas.csv")]
}

df_resumen_limpieza = pd.DataFrame(resumen)
ruta_csv_resumen = os.path.join(REPORTES_DIR, "resumen_limpieza_dataset.csv")
df_resumen_limpieza.to_csv(ruta_csv_resumen, index=False, encoding="utf-8")

print("Resumen guardado en:", ruta_csv_resumen)
display(df_resumen_limpieza)

print("\n=== Verificación posterior ===")

if not EJECUTAR_LIMPIEZA:
    print("La limpieza no fue aplicada porque EJECUTAR_LIMPIEZA está en False.")
else:
    if len(df_descartadas) == 0:
        print("No había imágenes inválidas para eliminar.")
    elif invalidas_aun_presentes == 0:
        print("OK: las imágenes inválidas detectadas ya no están en el dataset.")
    else:
        print("ADVERTENCIA: aún quedan imágenes inválidas detectadas dentro del dataset:", invalidas_aun_presentes)

    if len(df_duplicados) == 0:
        print("No había duplicados idénticos por SHA256 para eliminar.")
    elif duplicados_aun_presentes == 0:
        print("OK: los duplicados marcados para retirar ya no están en el dataset.")
    else:
        print("ADVERTENCIA: aún quedan duplicados marcados para retirar dentro del dataset:", duplicados_aun_presentes)

    if len(df_duplicados) > 0:
        print("Copias conservadas que siguen existiendo:", copias_conservadas_presentes)



In [ ]:
#Conteo dataset Test

import os
dataset_path = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO/Test"

total = 0
conteo = {}

for carpeta in sorted(os.listdir(dataset_path)):
    ruta_carpeta = os.path.join(dataset_path, carpeta)
    if os.path.isdir(ruta_carpeta):
        imagenes = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        conteo[carpeta] = len(imagenes)
        total += len(imagenes)

print(f"{'Emoción':<15} {'Cantidad':>10} {'Proporción':>12}")
print("-" * 40)
for emocion, cantidad in sorted(conteo.items()):
    print(f"{emocion:<15} {cantidad:>10} {(cantidad/total*100):>11.2f}%")
print("-" * 40)
print(f"{'TOTAL':<15} {total:>10} {'100.00%':>12}")

Emoción           Cantidad   Proporción
----------------------------------------
Anger                 1718       11.83%
Contempt              1312        9.04%
disgust               1248        8.60%
fear                  1664       11.46%
happy                 2704       18.63%
neutral               2368       16.31%
sad                   1584       10.91%
surprise              1920       13.22%
----------------------------------------
TOTAL                14518      100.00%


In [ ]:
#Conteo dataset Train

import os
dataset_path = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO/Train"

total = 0
conteo = {}

for carpeta in sorted(os.listdir(dataset_path)):
    ruta_carpeta = os.path.join(dataset_path, carpeta)
    if os.path.isdir(ruta_carpeta):
        imagenes = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        conteo[carpeta] = len(imagenes)
        total += len(imagenes)

print(f"{'Emoción':<15} {'Cantidad':>10} {'Proporción':>12}")
print("-" * 40)
for emocion, cantidad in sorted(conteo.items()):
    print(f"{emocion:<15} {cantidad:>10} {(cantidad/total*100):>11.2f}%")
print("-" * 40)
print(f"{'TOTAL':<15} {total:>10} {'100.00%':>12}")

Emoción           Cantidad   Proporción
----------------------------------------
anger                 1500       11.80%
contempt              1559       12.27%
disgust               1229        9.67%
fear                  1512       11.90%
happy                 2340       18.41%
neutral                  0        0.00%
sad                   3091       24.32%
surprise              1479       11.64%
----------------------------------------
TOTAL                12710      100.00%


In [ ]:
import os
base_path = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO"

conteo = {"Train": {}, "Test": {}}

for split in ["Train", "Test"]:
    dataset_path = os.path.join(base_path, split)
    for carpeta in sorted(os.listdir(dataset_path)):
        ruta_carpeta = os.path.join(dataset_path, carpeta)
        if os.path.isdir(ruta_carpeta):
            imagenes = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            conteo[split][carpeta.lower()] = len(imagenes)

total_train = sum(conteo["Train"].values())
total_test = sum(conteo["Test"].values())
total_general = total_train + total_test

emociones = sorted(set(conteo["Train"].keys()) | set(conteo["Test"].keys()))

print(f"{'Emoción':<15} {'Train':>10} {'Test':>10} {'Total':>10} {'Proporción':>12}")
print("-" * 60)
for emocion in emociones:
    t = conteo["Train"].get(emocion, 0)
    te = conteo["Test"].get(emocion, 0)
    total_emocion = t + te
    print(f"{emocion:<15} {t:>10} {te:>10} {total_emocion:>10} {(total_emocion/total_general*100):>11.2f}%")
print("-" * 60)
print(f"{'TOTAL':<15} {total_train:>10} {total_test:>10} {total_general:>10} {'100.00%':>12}")

Emoción              Train       Test      Total   Proporción
------------------------------------------------------------
anger                 1500       1718       3218       11.82%
contempt              1559       1312       2871       10.54%
disgust               1229       1248       2477        9.10%
fear                  1512       1664       3176       11.66%
happy                 2340       2704       5044       18.53%
neutral                  0       2368       2368        8.70%
sad                   3091       1584       4675       17.17%
surprise              1479       1920       3399       12.48%
------------------------------------------------------------
TOTAL                12710      14518      27228      100.00%


In [ ]:
# ── CONFIGURACIÓN ─────────────────────────────────────────────────────────────
BASE_PATH   = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO"
LABELS_CSV  = f"{BASE_PATH}/labels.csv"
SPLITS      = ["Train", "Test"]
IMG_EXT     = ('.jpg', '.jpeg', '.png')
# ──────────────────────────────────────────────────────────────────────────────

import os, pandas as pd

# 1. Leer labels.csv (detecta separador automáticamente)
df = pd.read_csv(LABELS_CSV, sep=None, engine="python")
print(f"Columnas detectadas: {df.columns.tolist()}")
print(f"Total filas en labels.csv: {len(df)}\n")

# pth tiene formato "emocion/imageXXXXXX" sin split ni extensión
# Se construye un índice de todos los archivos en disco para búsqueda rápida

# 2. Índice de disco: clave = (split, emocion, stem) → nombre real con extensión
print("Construyendo índice de disco...")
indice_disco = {}   # (split, label, stem) -> ruta completa
conteo_disco = {s: {} for s in SPLITS}

for split in SPLITS:
    split_path = os.path.join(BASE_PATH, split)
    for carpeta in sorted(os.listdir(split_path)):
        ruta_carpeta = os.path.join(split_path, carpeta)
        if not os.path.isdir(ruta_carpeta):
            continue
        label_lower = carpeta.lower()
        imgs = [f for f in os.listdir(ruta_carpeta) if f.lower().endswith(IMG_EXT)]
        conteo_disco[split][label_lower] = len(imgs)
        for img in imgs:
            stem = os.path.splitext(img)[0]   # nombre sin extensión
            indice_disco[(split, label_lower, stem)] = os.path.join(ruta_carpeta, img)

print(f"Archivos indexados: {len(indice_disco)}\n")

# 3. Conteo en disco
print("=== Conteo real en disco ===")
disco_df = pd.DataFrame(conteo_disco).fillna(0).astype(int)
disco_df["Total_disco"] = disco_df.sum(axis=1)
print(disco_df.to_string())
print(f"\nTOTAL disco → {disco_df['Total_disco'].sum()}\n")

# 4. Conteo según labels.csv (por label únicamente, sin split)
print("=== Conteo según labels.csv ===")
print(df["label"].value_counts().sort_index().to_string())
print(f"\nTOTAL csv → {len(df)}\n")

# 5. Cruce: buscar cada entrada del CSV en el índice de disco (en Train O Test)
print("=== Cruce labels.csv vs disco ===")
encontrados = {"Train": 0, "Test": 0}
no_encontrados = []

for _, row in df.iterrows():
    pth      = str(row["pth"])                      # "anger/image000000"
    label    = str(row["label"]).lower()
    stem     = os.path.splitext(os.path.basename(pth))[0]  # "image000000"

    hallado = False
    for split in SPLITS:
        if (split, label, stem) in indice_disco:
            encontrados[split] += 1
            hallado = True
            break
    if not hallado:
        no_encontrados.append({"label": row["label"], "pth": pth})

print(f"Encontrados en Train : {encontrados['Train']}")
print(f"Encontrados en Test  : {encontrados['Test']}")
print(f"NO encontrados       : {len(no_encontrados)}\n")

if no_encontrados:
    falt_df = pd.DataFrame(no_encontrados)
    print("Faltantes por emoción:")
    print(falt_df["label"].value_counts().sort_index().to_string())
    print("\nEjemplos de entradas no encontradas:")
    print(falt_df.head(8).to_string(index=False))
else:
    print("Todos los registros del CSV tienen su archivo en disco.")

Columnas detectadas: ['Unnamed: 0', 'pth', 'label', 'relFCs']
Total filas en labels.csv: 28175

Construyendo índice de disco...
Archivos indexados: 27228

=== Conteo real en disco ===
          Train  Test  Total_disco
anger      1500  1718         3218
contempt   1559  1312         2871
disgust    1229  1248         2477
fear       1512  1664         3176
happy      2340  2704         5044
neutral       0  2368         2368
sad        3091  1584         4675
surprise   1479  1920         3399

TOTAL disco → 27228

=== Conteo según labels.csv ===
label
anger       3608
contempt    3244
disgust     3472
fear        3043
happy       4336
neutral     2861
sad         2995
surprise    4616

TOTAL csv → 28175

=== Cruce labels.csv vs disco ===
Encontrados en Train : 8316
Encontrados en Test  : 8273
NO encontrados       : 11586

Faltantes por emoción:
label
anger       1610
contempt    1381
disgust     1674
fear        1076
happy        450
neutral     1983
sad         1266
surprise    2146


In [ ]:
# [Ivan Diaz 06/06/2026] — Split estratificado 70/15/15 por clase y guarda labels_split.csv

import pandas as pd
from sklearn.model_selection import train_test_split

# --- CONFIGURACIÓN ---
BASE_PATH = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO"
INPUT_CSV  = f"{BASE_PATH}/labels.csv"
OUTPUT_CSV = f"{BASE_PATH}/labels_split.csv"

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
RANDOM_SEED = 42
# ---------------------

df = pd.read_csv(INPUT_CSV, index_col=0)

splits = []

for label, grupo in df.groupby("label"):
    # Separar 70% train | 30% restante
    train, temp = train_test_split(
        grupo,
        test_size=(VAL_RATIO + TEST_RATIO),
        random_state=RANDOM_SEED,
        shuffle=True
    )
    # Del 30% restante: 50% val | 50% test  →  15% y 15% del total
    val, test = train_test_split(
        temp,
        test_size=0.5,
        random_state=RANDOM_SEED,
        shuffle=True
    )

    train = train.copy(); train["split"] = "train"
    val   = val.copy();   val["split"]   = "val"
    test  = test.copy();  test["split"]  = "test"

    splits.extend([train, val, test])

df_split = pd.concat(splits).sort_index()

df_split.to_csv(OUTPUT_CSV)

# Verificación
print(f"CSV guardado en: {OUTPUT_CSV}\n")
print(f"{'Label':<12} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 48)
resumen = df_split.groupby(["label", "split"]).size().unstack(fill_value=0)
for label, row in resumen.iterrows():
    total = row.sum()
    print(f"{label:<12} {row.get('train',0):>8} {row.get('val',0):>8} {row.get('test',0):>8} {total:>8}")
print("-" * 48)
totales = resumen.sum()
print(f"{'TOTAL':<12} {totales.get('train',0):>8} {totales.get('val',0):>8} {totales.get('test',0):>8} {df_split.shape[0]:>8}")

CSV guardado en: /content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO/labels_split.csv

Label           Train      Val     Test    Total
------------------------------------------------
anger            2525      541      542     3608
contempt         2270      487      487     3244
disgust          2430      521      521     3472
fear             2130      456      457     3043
happy            3035      650      651     4336
neutral          2002      429      430     2861
sad              2096      449      450     2995
surprise         3231      692      693     4616
------------------------------------------------
TOTAL           19719     4225     4231    28175


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO"

print("Existe BASE_PATH:", os.path.exists(BASE_PATH))
print("Existe labels_split.csv:", os.path.exists(f"{BASE_PATH}/labels_split.csv"))
print("Existe Train:", os.path.exists(f"{BASE_PATH}/Train"))
print("Existe Test:", os.path.exists(f"{BASE_PATH}/Test"))

Existe BASE_PATH: True
Existe labels_split.csv: True
Existe Train: True
Existe Test: True


In [ ]:
# [Rodrigo Donoso] [07/06] — Balanceo de clases en Train con sobremuestreo y augmentación

import os
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance

# =========================
# CONFIGURACIÓN
# =========================
BASE_PATH = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO"
INPUT_CSV = f"{BASE_PATH}/labels_split.csv"
OUTPUT_CSV = f"{BASE_PATH}/labels_split_balanced.csv"
BALANCED_DIR = f"{BASE_PATH}/Train_balanced"

RANDOM_SEED = 42
IMG_EXTS = (".jpg", ".jpeg", ".png")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(BALANCED_DIR, exist_ok=True)

# =========================
# CARGAR CSV
# =========================
df = pd.read_csv(INPUT_CSV)

df["label"] = df["label"].astype(str).str.lower().str.strip()
df["split"] = df["split"].astype(str).str.lower().str.strip()

train = df[df["split"] == "train"].copy()
val = df[df["split"] == "val"].copy()
test = df[df["split"] == "test"].copy()

print("=== Conteo original Train ===")
print(train["label"].value_counts().sort_index().to_string())
print()

# =========================
# FUNCIÓN PARA BUSCAR RUTA REAL DE LA IMAGEN
# =========================
def resolver_ruta_imagen(row):
    pth = str(row["pth"]).replace("\\", "/").strip()
    label = str(row["label"]).lower().strip()
    nombre_archivo = os.path.basename(pth)
    nombre_base = os.path.splitext(nombre_archivo)[0]

    candidatos = [
        os.path.join(BASE_PATH, "Train", label, nombre_archivo),
        os.path.join(BASE_PATH, "Test", label, nombre_archivo),
        os.path.join(BASE_PATH, "Train", nombre_archivo),
        os.path.join(BASE_PATH, "Test", nombre_archivo),
    ]

    for ruta in candidatos:
        if os.path.exists(ruta):
            return ruta

    # Búsqueda por nombre base dentro de la carpeta de la clase
    for split_dir in ["Train", "Test"]:
        carpeta_clase = os.path.join(BASE_PATH, split_dir, label)
        if os.path.isdir(carpeta_clase):
            for archivo in os.listdir(carpeta_clase):
                if archivo.lower().endswith(IMG_EXTS):
                    if os.path.splitext(archivo)[0] == nombre_base:
                        return os.path.join(carpeta_clase, archivo)

    return None

# =========================
# FUNCIÓN DE AUGMENTACIÓN
# =========================
def augmentar_imagen(img):
    # espejo horizontal
    if random.random() < 0.5:
        img = ImageOps.mirror(img)

    # rotación leve
    if random.random() < 0.5:
        angulo = random.uniform(-12, 12)
        img = img.rotate(angulo, resample=Image.BILINEAR)

    # brillo
    if random.random() < 0.5:
        factor_brillo = random.uniform(0.85, 1.15)
        img = ImageEnhance.Brightness(img).enhance(factor_brillo)

    # contraste
    if random.random() < 0.5:
        factor_contraste = random.uniform(0.85, 1.15)
        img = ImageEnhance.Contrast(img).enhance(factor_contraste)

    return img

# =========================
# BALANCEO DE CLASES SOLO EN TRAIN
# =========================
conteo_train = train["label"].value_counts().sort_index()
max_count = int(conteo_train.max())

print("Clase objetivo para balancear:", max_count)
print()

bloques_train_balanceado = []
contador_aug = 0
errores_ruta = []

for clase in sorted(conteo_train.index):
    grupo = train[train["label"] == clase].copy().reset_index(drop=True)
    cantidad_actual = len(grupo)
    faltan = max_count - cantidad_actual

    # guardar originales
    bloques_train_balanceado.append(grupo)

    print(f"Clase: {clase} | actuales: {cantidad_actual} | faltan para balancear: {faltan}")

    if faltan > 0:
        indices_muestra = np.random.choice(grupo.index, size=faltan, replace=True)
        nuevas_filas = []

        carpeta_salida_clase = os.path.join(BALANCED_DIR, clase)
        os.makedirs(carpeta_salida_clase, exist_ok=True)

        for idx in indices_muestra:
            fila = grupo.loc[idx].copy()
            ruta_original = resolver_ruta_imagen(fila)

            if ruta_original is None:
                errores_ruta.append({
                    "label": clase,
                    "pth": fila["pth"]
                })
                continue

            try:
                img = Image.open(ruta_original).convert("RGB")
                img_aug = augmentar_imagen(img)

                nuevo_nombre = f"aug_{clase}_{contador_aug:07d}.jpg"
                ruta_guardado = os.path.join(carpeta_salida_clase, nuevo_nombre)

                img_aug.save(ruta_guardado, quality=95)

                fila["pth"] = f"Train_balanced/{clase}/{nuevo_nombre}"
                fila["split"] = "train"

                nuevas_filas.append(fila)
                contador_aug += 1

            except Exception:
                errores_ruta.append({
                    "label": clase,
                    "pth": fila["pth"]
                })
                continue

        if len(nuevas_filas) > 0:
            bloques_train_balanceado.append(pd.DataFrame(nuevas_filas))

train_balanceado = pd.concat(bloques_train_balanceado, ignore_index=True)

# unir train balanceado + val + test
df_balanceado = pd.concat([train_balanceado, val, test], ignore_index=True)

# guardar csv final
df_balanceado.to_csv(OUTPUT_CSV, index=False)

# =========================
# RESUMEN FINAL
# =========================
print()
print("=== Conteo final Train balanceado ===")
print(train_balanceado["label"].value_counts().sort_index().to_string())

print()
print("=== Conteo final por split y clase ===")
print(df_balanceado.groupby(["split", "label"]).size().unstack(fill_value=0).to_string())

print()
print("CSV balanceado guardado en:")
print(OUTPUT_CSV)

print()
print("Cantidad de imágenes augmentadas creadas:", contador_aug)

if len(errores_ruta) > 0:
    errores_df = pd.DataFrame(errores_ruta)
    errores_csv = f"{BASE_PATH}/errores_balanceo.csv"
    errores_df.to_csv(errores_csv, index=False)

    print()
    print("Se detectaron imágenes con problemas de ruta o lectura.")
    print("Errores guardados en:")
    print(errores_csv)
else:
    print()
    print("No hubo errores de rutas ni lectura de imágenes.")

=== Conteo original Train ===
label
anger       2525
contempt    2270
disgust     2430
fear        2130
happy       3035
neutral     2002
sad         2096
surprise    3231

Clase objetivo para balancear: 3231

Clase: anger | actuales: 2525 | faltan para balancear: 706
Clase: contempt | actuales: 2270 | faltan para balancear: 961
Clase: disgust | actuales: 2430 | faltan para balancear: 801
Clase: fear | actuales: 2130 | faltan para balancear: 1101
Clase: happy | actuales: 3035 | faltan para balancear: 196
Clase: neutral | actuales: 2002 | faltan para balancear: 1229
Clase: sad | actuales: 2096 | faltan para balancear: 1135
Clase: surprise | actuales: 3231 | faltan para balancear: 0

=== Conteo final Train balanceado ===
label
anger       2728
contempt    2590
disgust     2848
fear        2824
happy       3204
neutral     2372
sad         2761
surprise    3231

=== Conteo final por split y clase ===
label  anger  contempt  disgust  fear  happy  neutral   sad  surprise
split              